# Notebook 03 — Modelagem, Validação e Seleção do Modelo Final**Referência na monografia:** Seção 3.4 (Modelagem e Validação)**Etapas executadas:**1. Treinamento dos três algoritmos (§3.4.1)2. Validação cruzada estratificada 5-fold + Grid Search otimizando *Recall* (§3.4.2)3. Ajuste do limiar de decisão pela curva Precision-Recall (§3.4.3)4. Comparação, teste de McNemar e seleção do modelo final (§3.4.4)> **Atenção ao tempo de execução.** O Grid Search completo pode levar de 10 a 40 minutos> dependendo da máquina. A variável `GRID_REDUZIDO` permite uma execução rápida para> testar o fluxo antes de rodar a busca completa.**Saída gerada:** `outputs/modelos/modelos_treinados.pkl`

In [ ]:
import osimport pickleimport timeimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom xgboost import XGBClassifierfrom sklearn.model_selection import GridSearchCV, StratifiedKFoldfrom sklearn.metrics import (accuracy_score, precision_score, recall_score,                             f1_score, roc_auc_score, confusion_matrix,                             roc_curve, precision_recall_curve,                             classification_report)from statsmodels.stats.contingency_tables import mcnemarwarnings.filterwarnings('ignore')RANDOM_STATE = 42np.random.seed(RANDOM_STATE)# Alterne para False quando for rodar a busca completa e definitivaGRID_REDUZIDO = TrueBASE_DIR = os.path.dirname(os.getcwd())DATA_DIR = os.path.join(BASE_DIR, 'data')FIG_DIR = os.path.join(BASE_DIR, 'outputs', 'figuras')TAB_DIR = os.path.join(BASE_DIR, 'outputs', 'tabelas')MOD_DIR = os.path.join(BASE_DIR, 'outputs', 'modelos')for d in (FIG_DIR, TAB_DIR, MOD_DIR):    os.makedirs(d, exist_ok=True)sns.set_theme(style='whitegrid', context='notebook')plt.rcParams['figure.dpi'] = 110plt.rcParams['savefig.dpi'] = 300plt.rcParams['savefig.bbox'] = 'tight'def salvar_fig(nome):    plt.savefig(os.path.join(FIG_DIR, f'{nome}.png'))with open(os.path.join(DATA_DIR, 'dados_processados.pkl'), 'rb') as f:    dados = pickle.load(f)X_treino = dados['X_treino_bal']y_treino = dados['y_treino_bal']X_teste = dados['X_teste_esc']y_teste = dados['y_teste']COLUNAS = dados['colunas']print(f'Treino (balanceado): {X_treino.shape}')print(f'Teste (original):    {X_teste.shape}')print(f'Modo de execução: {"GRID REDUZIDO (rápido)" if GRID_REDUZIDO else "GRID COMPLETO"}')

## 1. Definição dos algoritmos e do espaço de busca (§3.4.1 e §3.4.2)O espaço de busca reproduz a Tabela 3.2 da monografia.

In [ ]:
if GRID_REDUZIDO:    espaco_busca = {        'Regressão Logística': {            'modelo': LogisticRegression(max_iter=2000, penalty='l2',                                         random_state=RANDOM_STATE),            'grid': {'C': [0.1, 1, 10], 'solver': ['lbfgs']}        },        'Random Forest': {            'modelo': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),            'grid': {'n_estimators': [200], 'max_depth': [10, 20],                     'min_samples_leaf': [1, 5]}        },        'XGBoost': {            'modelo': XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1,                                    eval_metric='logloss'),            'grid': {'learning_rate': [0.1], 'n_estimators': [200],                     'max_depth': [3, 6], 'reg_lambda': [1, 10]}        },    }else:    # Espaço completo — corresponde exatamente à Tabela 3.2 da monografia    espaco_busca = {        'Regressão Logística': {            'modelo': LogisticRegression(max_iter=2000, penalty='l2',                                         random_state=RANDOM_STATE),            'grid': {'C': [0.01, 0.1, 1, 10, 100],                     'solver': ['lbfgs', 'liblinear']}        },        'Random Forest': {            'modelo': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),            'grid': {'n_estimators': [100, 200, 500],                     'max_depth': [5, 10, 20, None],                     'min_samples_leaf': [1, 2, 5]}        },        'XGBoost': {            'modelo': XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1,                                    eval_metric='logloss'),            'grid': {'learning_rate': [0.01, 0.1, 0.3],                     'n_estimators': [100, 200, 500],                     'max_depth': [3, 6, 10],                     'reg_lambda': [0, 1, 10]}        },    }total = sum(np.prod([len(v) for v in cfg['grid'].values()])            for cfg in espaco_busca.values())print(f'Combinações totais a avaliar: {int(total)}')print(f'Ajustes de modelo (x5 dobras): {int(total * 5)}')

## 2. Validação cruzada e otimização de hiperparâmetros (§3.4.2)`StratifiedKFold` com 5 dobras, otimizando o *Recall* — métrica central definida naSeção 2.4.4.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)modelos = {}resultados_cv = []for nome, cfg in espaco_busca.items():    print(f'\n{"=" * 60}')    print(f'Otimizando: {nome}')    print('=' * 60)    inicio = time.time()    busca = GridSearchCV(        estimator=cfg['modelo'],        param_grid=cfg['grid'],        scoring='recall',        cv=cv,        n_jobs=-1,        verbose=0,        refit=True    )    busca.fit(X_treino, y_treino)    duracao = time.time() - inicio    modelos[nome] = busca.best_estimator_    print(f'Melhores hiperparâmetros: {busca.best_params_}')    print(f'Recall médio na validação cruzada: {busca.best_score_:.4f}')    print(f'Tempo decorrido: {duracao:.1f}s')    resultados_cv.append({        'Modelo': nome,        'Recall (CV)': round(busca.best_score_, 4),        'Hiperparâmetros': str(busca.best_params_),        'Tempo (s)': round(duracao, 1)    })tabela_cv = pd.DataFrame(resultados_cv)display(tabela_cv)tabela_cv.to_csv(os.path.join(TAB_DIR, 'tab_resultados_cv.csv'), index=False)

## 3. Ajuste do limiar de decisão (§3.4.3)O limiar é selecionado maximizando o F1-Score sobre a curva Precision-Recall, calculada**no conjunto de treino via validação cruzada** — nunca no teste. Isso evita que o conjuntode avaliação realimente o processo de otimização.

In [ ]:
from sklearn.model_selection import cross_val_predictlimiares = {}for nome, modelo in modelos.items():    # Probabilidades out-of-fold: cada previsão vem de um modelo que não viu aquele dado    prob_oof = cross_val_predict(modelo, X_treino, y_treino, cv=cv,                                 method='predict_proba', n_jobs=-1)[:, 1]    prec, rec, thr = precision_recall_curve(y_treino, prob_oof)    f1 = 2 * (prec * rec) / (prec + rec + 1e-9)    idx = int(np.argmax(f1[:-1]))    limiares[nome] = float(thr[idx])    print(f'{nome:<22} limiar ótimo = {thr[idx]:.4f} '          f'(F1 = {f1[idx]:.4f}, Precisão = {prec[idx]:.4f}, Recall = {rec[idx]:.4f})')

In [ ]:
# Visualização das curvas Precision-Recallfig, axes = plt.subplots(1, 3, figsize=(17, 4.8))for ax, (nome, modelo) in zip(axes, modelos.items()):    prob_oof = cross_val_predict(modelo, X_treino, y_treino, cv=cv,                                 method='predict_proba', n_jobs=-1)[:, 1]    prec, rec, thr = precision_recall_curve(y_treino, prob_oof)    ax.plot(rec, prec, color='#2E86AB', linewidth=2)    f1 = 2 * (prec * rec) / (prec + rec + 1e-9)    idx = int(np.argmax(f1[:-1]))    ax.scatter(rec[idx], prec[idx], color='#C73E1D', s=90, zorder=5,               label=f'Limiar = {thr[idx]:.3f}')    ax.set_title(nome)    ax.set_xlabel('Recall')    ax.set_ylabel('Precisão')    ax.legend(loc='lower left')plt.suptitle('Curvas Precision-Recall e limiar selecionado (validação cruzada)', y=1.02)plt.tight_layout()salvar_fig('modelagem_curvas_precision_recall')plt.show()

## 4. Avaliação no conjunto de teste (§3.4.4)Todas as métricas são calculadas exclusivamente sobre o conjunto de teste, mantido em suadistribuição original desbalanceada.

In [ ]:
def avaliar(nome, modelo, limiar, X, y):    prob = modelo.predict_proba(X)[:, 1]    pred = (prob >= limiar).astype(int)    vn, fp, fn, vp = confusion_matrix(y, pred).ravel()    return {        'Modelo': nome,        'Limiar': round(limiar, 4),        'Acurácia': round(accuracy_score(y, pred), 4),        'Precisão': round(precision_score(y, pred), 4),        'Recall': round(recall_score(y, pred), 4),        'F1-Score': round(f1_score(y, pred), 4),        'AUC-ROC': round(roc_auc_score(y, prob), 4),        'VP': vp, 'VN': vn, 'FP': fp, 'FN': fn,    }, prob, predresultados, probs, preds = [], {}, {}for nome, modelo in modelos.items():    linha, prob, pred = avaliar(nome, modelo, limiares[nome], X_teste, y_teste)    resultados.append(linha)    probs[nome] = prob    preds[nome] = predcomparacao = pd.DataFrame(resultados).sort_values('Recall', ascending=False)print('DESEMPENHO NO CONJUNTO DE TESTE (ordenado por Recall)')display(comparacao)comparacao.to_csv(os.path.join(TAB_DIR, 'tab_comparacao_modelos.csv'), index=False)

In [ ]:
# Relatórios de classificação detalhadosfor nome in modelos:    print(f'\n{"=" * 60}')    print(nome)    print('=' * 60)    print(classification_report(y_teste, preds[nome],                                target_names=['Não-evasor', 'Evasor'],                                digits=4))

In [ ]:
# Matrizes de confusãofig, axes = plt.subplots(1, 3, figsize=(16, 4.5))for ax, nome in zip(axes, modelos):    mc = confusion_matrix(y_teste, preds[nome])    sns.heatmap(mc, annot=True, fmt='d', cmap='Blues', cbar=False,                xticklabels=['Não-evasor', 'Evasor'],                yticklabels=['Não-evasor', 'Evasor'], ax=ax)    ax.set_title(nome)    ax.set_xlabel('Classe prevista')    ax.set_ylabel('Classe real')plt.suptitle('Matrizes de confusão no conjunto de teste', y=1.03)plt.tight_layout()salvar_fig('modelagem_matrizes_confusao')plt.show()

In [ ]:
# Curvas ROC comparadasfig, ax = plt.subplots(figsize=(7.5, 6))cores = {'Regressão Logística': '#2E86AB', 'Random Forest': '#F18F01',         'XGBoost': '#C73E1D'}for nome in modelos:    fpr, tpr, _ = roc_curve(y_teste, probs[nome])    auc = roc_auc_score(y_teste, probs[nome])    ax.plot(fpr, tpr, linewidth=2, color=cores.get(nome),            label=f'{nome} (AUC = {auc:.4f})')ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Classificador aleatório')ax.set_xlabel('Taxa de Falsos Positivos')ax.set_ylabel('Taxa de Verdadeiros Positivos (Recall)')ax.set_title('Curvas ROC — conjunto de teste')ax.legend(loc='lower right')plt.tight_layout()salvar_fig('modelagem_curvas_roc')plt.show()

## 5. Teste de McNemar (§3.4.4)Avalia se a diferença de desempenho entre pares de classificadores é estatisticamentesignificativa ou fruto de variação amostral (Dietterich, 1998).

In [ ]:
from itertools import combinationsnomes = list(modelos.keys())testes = []for a, b in combinations(nomes, 2):    acertos_a = (preds[a] == y_teste.values)    acertos_b = (preds[b] == y_teste.values)    n00 = int(np.sum(~acertos_a & ~acertos_b))  # ambos erram    n01 = int(np.sum(~acertos_a & acertos_b))   # só B acerta    n10 = int(np.sum(acertos_a & ~acertos_b))   # só A acerta    n11 = int(np.sum(acertos_a & acertos_b))    # ambos acertam    tabela = [[n11, n10], [n01, n00]]    # exact=True quando as discordâncias são poucas; correção de continuidade caso contrário    usar_exato = (n01 + n10) < 25    res = mcnemar(tabela, exact=usar_exato, correction=not usar_exato)    testes.append({        'Comparação': f'{a} vs. {b}',        'Só A acerta': n10,        'Só B acerta': n01,        'Estatística': round(float(res.statistic), 4),        'p-valor': round(float(res.pvalue), 5),        'Significativo (α=0,05)': 'Sim' if res.pvalue < 0.05 else 'Não'    })tabela_mcnemar = pd.DataFrame(testes)display(tabela_mcnemar)tabela_mcnemar.to_csv(os.path.join(TAB_DIR, 'tab_teste_mcnemar.csv'), index=False)print('\nInterpretação: p < 0,05 indica que a diferença de desempenho entre os dois')print('classificadores é estatisticamente significativa.')

## 6. Seleção e persistência do modelo final

In [ ]:
modelo_final_nome = comparacao.iloc[0]['Modelo']modelo_final = modelos[modelo_final_nome]limiar_final = limiares[modelo_final_nome]print('=' * 60)print('MODELO FINAL SELECIONADO')print('=' * 60)print(f'Algoritmo: {modelo_final_nome}')print(f'Limiar de decisão: {limiar_final:.4f}')print(f'Recall no teste: {comparacao.iloc[0]["Recall"]:.4f}')print(f'AUC-ROC no teste: {comparacao.iloc[0]["AUC-ROC"]:.4f}')print(f'Falsos Negativos: {comparacao.iloc[0]["FN"]} '      f'(evasores não identificados)')artefatos = {    'modelos': modelos,    'limiares': limiares,    'modelo_final_nome': modelo_final_nome,    'modelo_final': modelo_final,    'limiar_final': limiar_final,    'comparacao': comparacao,    'probabilidades': probs,    'predicoes': preds,}caminho = os.path.join(MOD_DIR, 'modelos_treinados.pkl')with open(caminho, 'wb') as f:    pickle.dump(artefatos, f)print(f'\nArtefatos salvos em: {caminho}')print('Próxima etapa: Notebook 04 — Explicabilidade (SHAP e LIME)')